<a href="https://colab.research.google.com/github/MohinaRustamova/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohinaRustamova/lyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My Lane as an ML Task

**Task type: Ranking / scoring, built on a classifier's output.**

The lane question — "which pages should be reviewed first?" — is a ranking
question (framing-ml-problems maps "which ones first?" → ranking/scoring,
metric precision@K). But the underlying predictive piece is a binary
classifier: I'm predicting `is_declining_label` (declining vs. not) per
page, then using the model's predicted probability to sort the full
inventory into a ranked queue. Precision@K is the currency, not accuracy —
a reviewer only ever works from the top of the list, exactly as I argued
in ML-02.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/MohinaRustamova/flyrank-ml-internship", "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

Loaded: (30000, 44)


## 2. Target or Proxy

**What I'm predicting: `is_declining_label`** — this is 1 if the page's
traffic dropped more than 20% (comparing the last 30 days to the 30 days
before that), otherwise 0.

This is a "real" label, not something I made up — it comes straight from
actual traffic numbers, not a rule I invented. That's the safe kind of
label to use.

**One honest weak spot:** this label only tells me a page IS declining
right now — not that it WILL decline next month. A better version would
predict the future using only past data. I can't build that yet because my
current dataset is just one snapshot in time — I'd need the bigger
warehouse dataset (with multiple months) to do that properly. That's a
later-week upgrade, not this one.

Also: I will never use `trend_pct` or `trend_direction` as inputs to the
model — those are literally what the label is built from, so using them
would be cheating (the model would just be reading the answer key).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!pip install reportlab --quiet

import subprocess, sys
subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

import pandas as pd
df = pd.read_csv("data/processed/refresh_feature_vector.csv")
print("Loaded:", df.shape)

Loaded: (30000, 52)


In [13]:
print("is_declining_label" in df.columns)
df["is_declining_label"].value_counts(normalize=True)

True


,proportion
is_declining_label,
1,0.542067
0,0.457933


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success Metric

**The metric that actually matters for the business:** if a reviewer can
only check, say, 50 pages a week, my ranked list should give them more
real declining pages in those 50 spots than the old hand-written rule did.
In plain terms: fewer wasted checks, more real hits, same amount of time
spent.

**The metric I use to test the model:** Precision@K — "of the top K pages
my model picked, how many were actually declining?" I'll check this at
K=20 and K=50, since that's a realistic weekly workload.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
model = res["models"]["random_forest"]["precision_at_50"]
print(f"Old rule: {base:.3f}  |  Model: {model:.3f}  |  Model is {model/base:.1f}x better")


Old rule: 0.240  |  Model: 0.740  |  Model is 3.1x better


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. Unit of Analysis

One row = one page (`content_id`), looking at its last 90 days of data.
Same as what I used in ML-02.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df.shape)
print("is_declining_label" in df.columns)
lane_view = df[[
    "content_id", "content_type", "position_tier",
    "ctr", "impressions_90d", "is_declining_label"
]]
lane_view.head()

(30000, 52)
True


,content_id,content_type,position_tier,ctr,impressions_90d,is_declining_label
0,content_304f48230142,keyword article,striking,0.76,3803,1
1,content_a1fb4e703a9e,keyword article,page_3_5,0.05,15320,1
2,content_9aa793d4d895,keyword article,page_3_5,0.09,12581,1
3,content_331d6c4de07b,keyword article,page_1,0.49,11751,0
4,content_d99b7a2d90ca,keyword article,page_3_5,0.13,19140,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML Beats a Fixed Rule

I already have proof, not just a guess: in Notebook 02, my simple hand
rule (stale AND visible) got 0.240 Precision@50. The random forest got
0.740 — three times better, on the exact same test.

Why the simple rule loses: it only looks at two things with a hard
cutoff. But whether a page is "worth reviewing" really depends on several
things working together — content type, how deep it ranks, how engaging
it is, how long it is — and those things interact. For example, a thin
page might be totally fine at one ranking position but a real problem at
another. A basic if-statement can't capture that combo; a tree-based model
can.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# same numbers, used again to back this claim
print(f"Old rule vs. model: {base:.3f} → {model:.3f} ({model/base:.1f}x better)")

Old rule vs. model: 0.240 → 0.740 (3.1x better)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.